In [ ]:
import os
import sys
import glob
import gc
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from utils.paths import get_base_dir
from utils.helpers import get_binary_metrics_per_class

In [ ]:
current_dir = os.path.dirname(os.path.abspath(__file__))
parent_dir = os.path.dirname(os.path.dirname(current_dir)) 
sys.path.append(os.path.join(parent_dir, 'code'))

BASE_DIR = get_base_dir()
DATA_DIR = os.path.join(BASE_DIR, "data")
SAVE_DIR = str(BASE_DIR)

for d in ["models", "documentation"]:
    os.makedirs(os.path.join(SAVE_DIR, d), exist_ok=True)

CHUNKSIZE = 100_000
SAMPLE_FRAC_PER_CHUNK = 0.05
TOPK = 45 
RANDOM_STATE = 42
LABEL_COL = "label"
start_time = time.time()

In [ ]:
print("\n[1] Feature selection (chunked sampling)...")
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
if not csv_files:
    raise SystemExit(f"No CSV found in {DATA_DIR}")

sampled=[]
rows_seen=0
for fp in tqdm(csv_files, desc="Reading CSVs"):
    for c in pd.read_csv(fp, chunksize=CHUNKSIZE, low_memory=False):
        rows_seen += len(c)
        s = c.sample(frac=SAMPLE_FRAC_PER_CHUNK, random_state=RANDOM_STATE)
        sampled.append(s); del c, s; gc.collect()
data_sample = pd.concat(sampled, ignore_index=True)
print(f"Sampled {len(data_sample):,} rows from ~{rows_seen:,}")

X_s = data_sample.select_dtypes(include=[np.number]).drop(columns=[LABEL_COL], errors="ignore")
y_s = data_sample[LABEL_COL].astype(str).str.strip()
mi = mutual_info_classif(X_s.fillna(0), y_s, random_state=RANDOM_STATE)
top_idx = np.argsort(mi)[::-1][:TOPK]
selected_features = X_s.columns[top_idx].tolist()
print(f"Top {len(selected_features)} features: {selected_features[:8]} ...")
with open(os.path.join(SAVE_DIR, "models", "selected_features.txt"), "w") as f:
    f.write("\n".join(selected_features))
del sampled, data_sample, X_s, y_s, mi; gc.collect()

In [ ]:
print("\n[2] Streaming load & Interleaving data...")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
np.random.seed(RANDOM_STATE)
np.random.shuffle(csv_files)

if not csv_files:
    raise ValueError("No CSV files found!")

FEATURES_TO_DROP = ['dst_port'] 

selected_features = [f for f in selected_features if f not in FEATURES_TO_DROP]
print(f"  -> Explicitly dropped features: {FEATURES_TO_DROP}")

# 2. Prepare desired column list (Feature + Label)
wanted_cols = set(selected_features + [LABEL_COL])

data_list = []
total_rows = 0

for fp in tqdm(csv_files, desc="Reading CSVs"):
    try:
        # Read header to check what columns exist
        file_header = pd.read_csv(fp, nrows=0).columns.tolist()
        
        # Only take intersection between what we want and what file has
        actual_usecols = list(wanted_cols.intersection(file_header))
        
        # Skip file if missing label column
        if LABEL_COL not in actual_usecols:
            print(f"Warning: Skipping file {os.path.basename(fp)}: Missing label column '{LABEL_COL}'")
            continue

        # Read file with safe column list
        for chunk in pd.read_csv(fp, chunksize=CHUNKSIZE, usecols=actual_usecols, low_memory=False):
            # 1. Handle Label
            chunk[LABEL_COL] = chunk[LABEL_COL].astype(str).str.strip()
            
            # 2. Handle numeric data types
            for col in chunk.select_dtypes(include=["float64"]).columns:
                chunk[col] = pd.to_numeric(chunk[col], downcast="float")
            for col in chunk.select_dtypes(include=["int64"]).columns:
                chunk[col] = pd.to_numeric(chunk[col], downcast="integer")
                
            data_list.append(chunk)
            total_rows += len(chunk)
            
    except Exception as e:
        print(f"Error reading file {fp}: {e}")
        continue

print(f"  -> Total raw rows loaded: {total_rows:,}")

if not data_list:
    raise ValueError("No data loaded!")

# Concatenate
data = pd.concat(data_list, ignore_index=True, copy=False)
del data_list
gc.collect()

# Fill NaN values if files have inconsistent columns
data.fillna(0, inplace=True)

# IMPORTANT: DROP DUPLICATES BASED ON FEATURES
print(f"  -> Rows before removing duplicates: {len(data):,}")

feature_cols_only = [c for c in data.columns if c != LABEL_COL]

# Drop duplicate
data.drop_duplicates(subset=feature_cols_only, keep='first', inplace=True)

print(f"  -> Rows after removing duplicates:  {len(data):,}")

# Shuffle
print("  -> Interleaving (Shuffling) all data...")
data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Update selected_features based on actual data
selected_features = [c for c in data.columns if c != LABEL_COL]
print(f"Final features list updated ({len(selected_features)} features).")

print(f"Data loaded, cleaned & shuffled. Shape: {data.shape}")


In [ ]:
print("\n[3] Label encoding...")
data[LABEL_COL] = data[LABEL_COL].astype(str).str.strip().str.replace(r"[\s\-]+", "_", regex=True)
unique_lbl = sorted(data[LABEL_COL].unique())
mapping = {lbl: i for i, lbl in enumerate(unique_lbl)}
data["label_encoded"] = data[LABEL_COL].map(mapping).astype("int32")
data[LABEL_COL] = data["label_encoded"]
data.drop(columns=["label_encoded"], inplace=True)
with open(os.path.join(SAVE_DIR, "models", "label_mapping.pkl"), "wb") as f:
    pickle.dump(mapping, f)

# Save mapping as CSV for reference
print(f"Saving label mapping to CSV...")
mapping_csv_path = os.path.join(SAVE_DIR, "documentation", "label_mapping.csv")
try:
    pd.DataFrame(list(mapping.items()), columns=["LabelName", "EncodedValue"]) \
      .sort_values("EncodedValue") \
      .to_csv(mapping_csv_path, index=False)
    print(f"Saved label mapping (csv) to {mapping_csv_path}")
except Exception as e:
    print(f"Could not save label mapping csv: {e}")
print(f"Encoded {len(mapping)} labels.")

# Define 'classes' here, right after encoding
classes = np.array(sorted(data[LABEL_COL].unique()))
print(f"Defined {len(classes)} classes globally.")

# Save 'classes' for deployment if needed
with open(os.path.join(SAVE_DIR, "models" ,"classes.pkl"), "wb") as f:
    pickle.dump(classes, f)


In [ ]:
print("\n[4] Splitting, Scaling & Injecting Noise...")

# [4.1] Separate Features and Labels
feature_names_list = [col for col in data.columns if col != LABEL_COL]
y_series = data.pop(LABEL_COL)
X_df = data
del data
gc.collect()

# [4.2] Split Train/Test (80/20)
print(f"  -> Splitting Train/Test (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_series, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y_series 
)

# [4.3] Scaling (StandardScaler)
print("  -> Fitting scaler on Train set...")
scaler = StandardScaler()

# Fit & Transform Train
X_train = scaler.fit_transform(X_train)
# Transform Test (only transform, no fit, no noise)
X_test = scaler.transform(X_test)

# Convert y to numpy for processing
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

split_summary = f"""
Train features: {X_train.shape} (With Feature & Label Noise)
Train labels: {y_train.shape}
Test features: {X_test.shape} (Clean)
Test labels: {y_test.shape}
"""
print(split_summary)

# [4.4] Saving Artifacts
print("\n[4.4] Saving artifacts...")
with open(os.path.join(SAVE_DIR, "models", "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
with open(os.path.join(SAVE_DIR, "models", "feature_names.pkl"), "wb") as f:
    pickle.dump(feature_names_list, f)
inv_mapping = {v: k for k, v in mapping.items()}
with open(os.path.join(SAVE_DIR, "models", "label_mapping_inverse.pkl"), "wb") as f:
    pickle.dump(inv_mapping, f)

print("Data preparation complete.")


In [ ]:
print("\n[5] Training & Comparing Multiple Models...")
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import datetime

# --- MODEL CONFIGURATION ---
models_to_run = {
    "LightGBM": LGBMClassifier(
        boosting_type="gbdt", objective="multiclass", num_class=len(classes), 
        n_estimators=1000, learning_rate=0.01, num_leaves=7, max_depth=5, 
        reg_alpha=5.0, reg_lambda=10.0, subsample=0.8, colsample_bytree=0.5, 
        min_child_samples=3000, class_weight='balanced', 
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=15, max_features='sqrt', min_samples_leaf=5, 
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=1000, learning_rate=0.01, max_depth=5, reg_alpha=5.0, 
        reg_lambda=10.0, subsample=0.8, colsample_bytree=0.5, min_child_weight=30, 
        class_weight='balanced', n_jobs=-1, objective="multi:softmax", 
        num_class=len(classes), use_label_encoder=False, eval_metric='mlogloss', 
        random_state=RANDOM_STATE
    ),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=10, class_weight='balanced', random_state=RANDOM_STATE
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=500, C=0.1, penalty='l2', class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    ),
    "NaiveBayes": GaussianNB()    
}

results = []
feature_names = feature_names_list

# --- TRAINING LOOP ---
for name, model in models_to_run.items():
    print(f"\nTraining {name}...")
    t0 = time.time()
    
    # Fit model
    if name == "LightGBM":
        model.fit(X_train, y_train, feature_name=feature_names, categorical_feature=None)
    else:
        model.fit(X_train, y_train)
        
    y_pred = model.predict(X_test)

    # Compute metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    runtime = round(time.time() - t0, 2)
    print(f"{name}: Accuracy={acc:.4f}, F1={f1:.4f}, Time={runtime}s")

    # Save comparison results
    results.append({
        "Model": name, "Accuracy": acc, "F1_macro": f1, 
        "Runtime_sec": runtime, "Train_Timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

    # 1. SAVE MODEL
    model_path = os.path.join(SAVE_DIR, "models", f"{name}_model.pkl")
    with open(model_path, "wb") as f:
        pickle.dump({
            "model": model, "scaler": scaler, 
            "features": feature_names, "mapping": mapping
        }, f)
    print(f"Saved Model to {model_path}")

    # 2. SAVE CONFUSION MATRIX
    cm = confusion_matrix(y_test, y_pred, normalize="true")
    cm_path = os.path.join(SAVE_DIR, "documentation", f"conf_matrix_{name}.csv")
    pd.DataFrame(cm).to_csv(cm_path, index=False)

    # 3. SAVE CLASSIFICATION REPORT
    try:
        inv_mapping = {v: k for k, v in mapping.items()}
        target_names = [inv_mapping[i] for i in range(len(mapping))]
        report = classification_report(y_test, y_pred, target_names=target_names, zero_division=0)
        
        report_path = os.path.join(SAVE_DIR, "documentation", f"classification_report_{name}.txt")
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(f"CLASSIFICATION REPORT ({name})\n")
            f.write(report)
        print(f"Saved Classification Report")
    except Exception as e:
        print(f"Report error: {e}")

    # 4. CALCULATE & SAVE BINARY METRICS (ONE-VS-REST)
    print(f"Calculating Binary Metrics for {name}...")
    binary_df = get_binary_metrics_per_class(y_test, y_pred, mapping)
    
    print(f"Lowest F1 Scores for {name}:")
    print(binary_df.sort_values("F1_Score").head(5)[["Label_Name", "F1_Score", "Precision", "Recall"]].to_string(index=False))
    
    binary_report_path = os.path.join(SAVE_DIR, "documentation", f"binary_metrics_{name}.csv")
    binary_df.to_csv(binary_report_path, index=False)

    # 5. PLOT & SAVE FEATURE IMPORTANCE (Only for Tree-based Models)
    if name in ["LightGBM", "RandomForest", "XGBoost", "DecisionTree"]:
        try:
            print(f"Generating Feature Importance for {name}...")
            
            importances = None
            if name == "LightGBM":
                importances = model.booster_.feature_importance(importance_type='gain')
            elif hasattr(model, "feature_importances_"):
                importances = model.feature_importances_
            
            if importances is not None:
                fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
                fi_df = fi_df.sort_values(by='importance', ascending=False)
                fi_df.to_csv(os.path.join(SAVE_DIR, "documentation", f"feature_importance_{name}.csv"), index=False)

                plt.figure(figsize=(10, 8))
                top_20 = fi_df.head(20)
                plt.barh(top_20['feature'], top_20['importance'], color='skyblue')
                plt.xlabel("Importance")
                plt.title(f"Feature Importance - {name}")
                plt.gca().invert_yaxis()
                plt.tight_layout()
                plt.savefig(os.path.join(SAVE_DIR, "documentation", f"feature_importance_{name}.png"))
                plt.close()
        except Exception as e:
            print(f"Could not plot feature importance for {name}: {e}")


In [ ]:
res = pd.DataFrame(results).sort_values("F1_macro", ascending=False)
res_path = os.path.join(SAVE_DIR, "documentation", "model_comparison_summary.csv")
res.to_csv(res_path, index=False)
print("\nSummary:\n", res)
print(f"\nSaved model comparison summary to: {res_path}")

end_time = time.time()
total_runtime = round(end_time - start_time, 2)
print(f"\nPipeline started at: {datetime.datetime.fromtimestamp(start_time)}")
print(f"Finished at: {datetime.datetime.fromtimestamp(end_time)}")
print(f"Total runtime: {total_runtime/60:.2f} minutes")

# SAVE PIPELINE SUMMARY REPORT
print(f"Saving pipeline summary report...")
pipeline_summary_path = os.path.join(SAVE_DIR, "documentation", "pipeline_summary.txt")
try:
    with open(pipeline_summary_path, "w", encoding="utf-8") as f:
        f.write("PIPELINE SUMMARY\n")
        f.write(f"Pipeline Start: {datetime.datetime.fromtimestamp(start_time)}\n")
        f.write(f"Pipeline End: {datetime.datetime.fromtimestamp(end_time)}\n")
        f.write(f"Total Runtime (sec): {total_runtime}\n")
        f.write(f"Total Runtime (min): {total_runtime/60:.2f}\n")
        
        f.write("\nData Split\n")
        f.write(split_summary)

        best_model_stats = res.iloc[0]
        f.write("\nBest Model Stats (from summary)\n")
        f.write(f"Model: {best_model_stats['Model']}\n")
        f.write(f"F1_macro: {best_model_stats['F1_macro']:.6f}\n")
        f.write(f"Accuracy: {best_model_stats['Accuracy']:.6f}\n")
        f.write(f"Precision: {best_model_stats['Precision']:.6f}\n")
        f.write(f"Recall: {best_model_stats['Recall']:.6f}\n")
        f.write(f"Runtime_sec: {best_model_stats['Runtime_sec']}\n")
        
    print(f"Saved all pipeline stats to: {pipeline_summary_path}")
except Exception as e:
    print(f"Could not save pipeline summary: {e}")

print("\nTraining pipeline completed successfully.")
